# AI-Driven Fraud Detection In Public Financial Management Using Convolutional Neural Networks

## Libraries

In [2]:
# ================ Import Libraries ====================
import os
import time
import uuid
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import logging; logging.basicConfig(level=logging.INFO)
import joblib
import keras_tuner as kt
import json
import shap
import tensorflow as tf
from tensorflow.keras.models import clone_model
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    auc as roc_auc,
    confusion_matrix,
    recall_score,
    precision_score,
    accuracy_score,
    f1_score,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score

)
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K
from sklearn.ensemble import RandomForestClassifier; 
from xgboost import XGBClassifier; 
from scipy.stats import wilcoxon
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid



# Reproducibility Setup

In [5]:
# Reproducibility Setup

def set_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    # For older TF versions:
    # tf.keras.utils.set_random_seed(seed)

# Set the seeds
set_seeds(42)



# Model Building and Hyperparameter Tuning

In [7]:
# ================== Model Building and Hyperparameter Tuning ==========================

def build_tunable_model(hp, input_shape):
    model = Sequential()
    
    # Modern way: explicit Input layer (removes warning)
    model.add(Input(shape=input_shape))
    
    # First Conv layer (no input_shape here)
    model.add(Conv1D(
        filters=hp.Choice('filters_0', [32, 64]),
        kernel_size=hp.Int('kernel_size_0', 1, 2),
        activation='relu',
        padding='same'
    ))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(
        pool_size=hp.Int('pool_0', 1, 1)
    ))
    model.add(Dropout(hp.Float('dropout_0', 0.2, 0.5)))
    
    # Second Conv (optional)
    if hp.Boolean('add_second_conv'):
        model.add(Conv1D(
            filters=hp.Choice('filters_1', [32, 64]),
            kernel_size=1,
            activation='relu',
            padding='same'
        ))
        model.add(BatchNormalization())
        model.add(Dropout(hp.Float('dropout_1', 0.2, 0.5)))
    
    model.add(Flatten())
    
    for i in range(hp.Int('num_dense_layers', 1, 2)):
        model.add(Dense(
            units=hp.Choice(f'units_{i}', [64, 128]),
            activation='relu',
            kernel_regularizer=l2(hp.Float('l2_reg', 1e-4, 1e-2))
        ))
        model.add(Dropout(hp.Float(f'dense_dropout_{i}', 0.3, 0.5)))
    
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = hp.Choice('optimizer', ['adam', 'rmsprop'])
    lr = hp.Float('lr', 1e-4, 1e-2)
    if optimizer == 'adam':
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='pr_auc', curve='PR')]
    )
    return model

# ============= Optimize Model ===================
def optimize_model(X_train, y_train, input_shape):
    """hyperparameter optimization with proper validation handling"""
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train, y_train, 
        test_size=0.2, 
        stratify=y_train,
        random_state=42
    )
    
    smote = SMOTE(sampling_strategy=0.5, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(
        X_train_split.reshape(X_train_split.shape[0], -1),
        y_train_split
    )
    X_train_res = X_train_res.reshape(-1, *input_shape)
    
    print(f"Class distribution - Train: {np.bincount(y_train_res)}, Val: {np.bincount(y_val)}")
    
    tuner = kt.Hyperband(
        lambda hp: build_tunable_model(hp, input_shape),
        objective=kt.Objective("val_pr_auc", direction="max"),
        max_epochs=30,
        factor=3,
        directory='nigeria_tuning',
        project_name='payroll_fraud',
        overwrite=True
    )
    
    tuner.search(
        X_train_res, y_train_res,
        validation_data=(X_val, y_val),
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_pr_auc',
                patience=5,
                mode='max',
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_pr_auc',
                factor=0.5,
                patience=3,
                verbose=1
            )
        ],
        class_weight={0: 1, 1: 10},
        batch_size=64,
        epochs=50,
        verbose=2
    )
    
    # Get the best hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    #print("\nBest Hyperparameters found by KerasTuner:")
    #print(best_hps.values)
    logging.info("Best Hyperparameters:")
    for key, value in best_hps.values.items():
        logging.info(f"{key}: {value}")
    # model.summary()  # Prints layer details

    # Return both the best model AND the best hyperparameters
    return tuner.get_best_models(num_models=1)[0], best_hps
    


# Cross-Validation and Final Model Training

In [10]:
# ================ Pre-configured model for cross-validation - now can be dynamic or static =================
def build_model_with_hps(hps, input_shape):
    """Builds a model using the provided HyperParameters or a static configuration."""
    model = Sequential()

    # Use hyperparameter values if provided
    filters_0 = hps.get('filters_0', 64) # Default to 64 if not tuned
    kernel_size_0 = hps.get('kernel_size_0', 2) # Default to 2
    pool_0 = hps.get('pool_0', 1) # Default to 1
    dropout_0 = hps.get('dropout_0', 0.3) # Default to 0.3
    model.add(Input(shape=input_shape))
    model.add(Conv1D(
        filters=filters_0,
        kernel_size=kernel_size_0,
        activation='relu',
        padding='same'
        #input_shape=input_shape
    ))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=pool_0))
    model.add(Dropout(dropout_0))

    if hps.get('add_second_conv', True): # Default to adding second conv
        filters_1 = hps.get('filters_1', 128) # Default to 128
        dropout_1 = hps.get('dropout_1', 0.4) # Default to 0.4
        model.add(Conv1D(
            filters=filters_1,
            kernel_size=2, # Static for second conv for simplicity or set by tuner if available
            activation='relu',
            padding='same'
        ))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_1))
    
    model.add(Flatten())
    
    num_dense_layers = hps.get('num_dense_layers', 2) # Default to 2
    for i in range(num_dense_layers):
        units_i = hps.get(f'units_{i}', 128 if i == 0 else 64) # Default units
        l2_reg = hps.get('l2_reg', 0.01) # Default l2 reg
        dense_dropout_i = hps.get(f'dense_dropout_{i}', 0.5 if i == 0 else 0.4) # Default dense dropout

        model.add(Dense(
            units=units_i,
            activation='relu',
            kernel_regularizer=l2(l2_reg)
        ))
        model.add(BatchNormalization() if i == 0 else Dropout(0)) # Add BN to first dense if needed
        model.add(Dropout(dense_dropout_i))
    
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer_choice = hps.get('optimizer', 'adam') # Default to adam
    lr = hps.get('lr', 0.001) # Default to 0.001
    
    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr, clipvalue=0.5)
    else: # rmsprop
        optimizer = RMSprop(learning_rate=lr, clipvalue=0.5)
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='pr_auc', curve='PR')]
    )
    
    return model

# ================== Apply Cross-Validation ===========================
def cross_validate_model(X, y, input_shape, n_splits=5, best_hps=None):
    """5-fold cross-validation with proper SMOTE application, using best_hps if provided"""
    print(f"\nStarting {n_splits}-fold cross-validation...")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_scores = []
    models = []
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold + 1}/{n_splits} ---")
        X_train_fold, X_val = X[train_idx], X[val_idx]
        y_train_fold, y_val = y[train_idx], y[val_idx]
        smote = SMOTE(sampling_strategy=0.5, random_state=42)
        X_res, y_res = smote.fit_resample(
            X_train_fold.reshape(X_train_fold.shape[0], -1),
            y_train_fold
        )
        X_res = X_res.reshape(-1, *input_shape)
        model = build_model_with_hps(best_hps if best_hps is not None else {}, input_shape)
        history = model.fit(
            X_res, y_res,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=64,
            class_weight={0: 1, 1: 10},
            callbacks=[...],  # ← complete this with your actual callbacks list
            verbose=0
        )
        val_pr_auc = max(history.history['val_pr_auc'])
        cv_scores.append(val_pr_auc)
        models.append(model)
        
        # NEW: Compute detailed metrics on this fold's validation set
        y_val_pred_prob = model.predict(X_val, verbose=0).flatten()
        optimal_thresh = 0.5  # or use your find_best_threshold logic here if desired
        y_val_pred = (y_val_pred_prob >= optimal_thresh).astype(int)
        fold_metrics.append({
            'fold': fold + 1,
            'PR-AUC': val_pr_auc,
            'precision': precision_score(y_val, y_val_pred, zero_division=0),
            'recall': recall_score(y_val, y_val_pred, zero_division=0),
            'f1': f1_score(y_val, y_val_pred, zero_division=0)
        })
        print(f"Fold {fold+1} PR-AUC: {val_pr_auc:.3f}")
        # ... your plots code here ...
    
    # After loop: Summarize
    print("\n--- Cross-Validation Results ---")
    metrics_df = pd.DataFrame(fold_metrics)
    print(metrics_df.round(4))
    metrics_df.to_csv('cv_detailed_metrics.csv', index=False)
    
    print(f"Mean PR-AUC: {metrics_df['PR-AUC'].mean():.4f} ± {metrics_df['PR-AUC'].std():.4f}")
    print(f"Mean Precision: {metrics_df['precision'].mean():.4f} ± {metrics_df['precision'].std():.4f}")
    print(f"Mean Recall: {metrics_df['recall'].mean():.4f} ± {metrics_df['recall'].std():.4f}")
    print(f"Mean F1: {metrics_df['f1'].mean():.4f} ± {metrics_df['f1'].std():.4f}")
    
    best_idx = np.argmax(cv_scores)
    return models[best_idx], cv_scores



# Model Baseline

In [13]:
def train_baselines(X, y, input_shape, cv_scores=None):
    # Flatten for tree models
    X_flat = X.reshape(X.shape[0], -1)
    baselines = {
        'RF': RandomForestClassifier(class_weight='balanced', random_state=42),
        'XGBoost': XGBClassifier(scale_pos_weight=10, random_state=42)
    }
    results = {}
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for name, clf in baselines.items():
        fold_scores = []
        for train_idx, val_idx in skf.split(X_flat, y):
            X_train, X_val = X_flat[train_idx], X_flat[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_val)
            fold_scores.append(f1_score(y_val, y_pred, zero_division=0))
        results[name] = {'mean_f1': np.mean(fold_scores), 'std_f1': np.std(fold_scores)}
        if cv_scores is not None:
            stat, p = wilcoxon(fold_scores, cv_scores)
            logging.info(f"{name} vs CNN Wilcoxon test: stat={stat:.2f}, p={p:.4f}")
            results[name]['wilcoxon_p'] = p
    return results
    
    

# Real-Time Fraud Detection Simulation and Dashboard

In [16]:
# Real-Time Fraud Detection Simulation and Dashboard
class FraudDetectionSystem:
    def __init__(self, model, df, scaler, categorical_cols, feature_cols, seed=42):
        self.model = model
        self.df = df
        self.scaler = scaler
        self.categorical_cols = categorical_cols
        self.feature_cols = feature_cols
        self.fraud_log = []
        self.alert_count = 0
        self.rng = np.random.RandomState(seed)
        
        if len(model.input_shape) != 3:
            raise ValueError("Model must expect 3D input (batch, timesteps, features)")
        _, self.expected_timesteps, self.expected_features = model.input_shape
        print(f"Model expects: {self.expected_timesteps} timesteps, {self.expected_features} features")
        
        # Pass the desired fraud rate to the creation method
        self.record_pool = self._create_augmented_pool(size_multiplier=10, fraud_rate=0.05)

    def _create_augmented_pool(self, size_multiplier=10, fraud_rate=0.01):
        """Creates an augmented data pool for the simulation, including known fraud cases."""
        pool = []
        df_fraud = self.df[self.df['IsFraudulent'] == 1].copy()
        df_non_fraud = self.df[self.df['IsFraudulent'] == 0].copy()
        
        if df_fraud.empty:
            print("Warning: No fraudulent records found in the original dataset.")
            num_fraud_records = 0
        else:
            total_records_to_add = len(self.df) * size_multiplier
            num_fraud_records = int(total_records_to_add * fraud_rate)
        
        num_non_fraud_records = (len(self.df) * size_multiplier) - num_fraud_records
        
        # Generate and augment non-fraud records
        for _ in range(num_non_fraud_records):
            trans = df_non_fraud.sample(1, random_state=self.rng).iloc[0].to_dict()
            if 'PaidSalary' in trans:
                trans['PaidSalary'] *= np.random.uniform(0.9, 1.1)
            if 'Age' in trans:
                trans['Age'] = np.clip(trans['Age'] + np.random.randint(-2, 3), 18, 70)
            if 'EmploymentStatus' in trans:
                trans['EmploymentStatus'] = np.random.choice(['Active', 'Inactive'], p=[0.95, 0.05])
            
            # Explicitly set 'IsFraudulent' to 0
            trans['IsFraudulent'] = 0
            pool.append(trans)
        
        # Generate and augment fraud records
        for _ in range(num_fraud_records):
            trans = df_fraud.sample(1, random_state=self.rng).iloc[0].to_dict()
            # Augment fraudulent transactions to make them more distinct
            trans['PaidSalary'] *= np.random.uniform(1.2, 1.5)
            trans['IsFraudulent'] = 1 # Keep the true label
            pool.append(trans)
            
        self.rng.shuffle(pool)
        return pool

    def generate_record(self):
        return random.choice(self.record_pool)

    def preprocess_record(self, record):
        try:
            if isinstance(record, dict):
                record_df = pd.DataFrame([record])
            else:
                record_df = record
            
            record_encoded = pd.get_dummies(record_df, columns=self.categorical_cols)
            
            for col in self.feature_cols:
                if col not in record_encoded.columns:
                    record_encoded[col] = 0
            
            record_aligned = record_encoded[self.feature_cols]
            scaled = self.scaler.transform(record_aligned)
            return scaled.reshape(1, self.expected_timesteps, self.expected_features)
        
        except Exception as e:
            print(f"Preprocessing error: {str(e)}")
            return np.zeros((1, self.expected_timesteps, self.expected_features))
            

    def send_alert(self, record, fraud_prob):
        self.alert_count += 1
        alert_html = f"""
        <div style='border:2px solid red; padding:10px; margin:10px;'>
            <h3>FRAUD ALERT #{self.alert_count}</h3>
            <p>Employee ID: {record.get('EmployeeID', 'N/A')}</p>
            <p>Name: {record.get('Name', 'N/A')}</p>
            <p>Department: {record.get('Department', 'N/A')}</p>
            <p>Grade Level: {record.get('GradeLevel', 'N/A')}</p>
            <p>Position: {record.get('Position', 'N/A')}</p>
            <p>Salary: ₦{record.get('PaidSalary', 0):,.2f}</p>
            <p>Fraud Probability: {fraud_prob:.2%}</p>
        </div>
        """
        display(HTML(alert_html))
    
    def create_dashboard(self):
        df_log = pd.DataFrame(self.fraud_log[-800:])
        
        # Summary metrics
        metrics_html = f"""
        <div style='display:flex; justify-content:space-between; margin-bottom:20px;'>
            <div style='border:1px solid #ddd; padding:10px; width:30%;'>
                <h4>Records Processed</h4>
                <p style='font-size:24px;'>{len(df_log)}</p>
            </div>
            <div style='border:1px solid #ddd; padding:10px; width:30%;'>
                <h4>Fraud Rate</h4>
                <p style='font-size:24px;'>{df_log['fraud_alert'].mean():.2%}</p>
            </div>
            <div style='border:1px solid #ddd; padding:10px; width:30%;'>
                <h4>Alerts Triggered</h4>
                <p style='font-size:24px;'>{self.alert_count}</p>
            </div>
        </div>
        """
        
        # Recent flagged cases
        flagged_df = df_log[df_log['fraud_alert'] == True]
        if not flagged_df.empty:
            flagged_html = "<h4>Recent Flagged Cases</h4><ul>"
            for _, row in flagged_df.tail(5).iterrows():
                flagged_html += f"<li>{row['timestamp']} - {row['EmployeeID']} - Prob: {row['fraud_probability']:.2%}</li>"
            flagged_html += "</ul>"
        else:
            flagged_html = "<p>No recent fraud cases detected</p>"
        
        # Create plot
        fig = px.line(df_log, x='timestamp', y='fraud_probability',
                      color='Department', template='plotly_dark',
                      title='Real-Time Fraud Probability Trend')
        
        return HTML(metrics_html + flagged_html), fig

    def process_record(self, record):
        try:
            processed = self.preprocess_record(record)
            if processed.shape[1:] != self.model.input_shape[1:]:
                raise ValueError(f"Shape mismatch: {processed.shape[1:]} vs {self.model.input_shape[1:]}")
            
            start_time = time.time()
            fraud_prob = float(self.model.predict(processed, verbose=0)[0][0])
            inference_time_ms = (time.time() - start_time) * 1000  # ms
            
            # Add to log
            self.fraud_log.append({
                **record,
                'timestamp': pd.Timestamp.now(),
                'fraud_probability': fraud_prob,
                'fraud_alert': fraud_prob > self.alert_threshold,  # Use class threshold
                'inference_time_ms': inference_time_ms
            })
            
            logging.info(f"Inference time: {inference_time_ms:.2f} ms")
            
            return fraud_prob
        
        except Exception as e:
            print(f"Processing error: {str(e)}")
            return 0.0
        
    def run_simulation(self, duration=300, interval=1, alert_threshold=0.95):
        start_time = time.time()          # Start timing the whole simulation
        processed = 0
    
        try:
            while time.time() - start_time < duration:
                record = self.generate_record()
                fraud_prob = self.process_record(record)
                
                self.fraud_log.append({
                    **record,
                    'timestamp': pd.Timestamp.now(),
                    'fraud_probability': fraud_prob,
                    'fraud_alert': fraud_prob > alert_threshold
                })
                processed += 1
                
                # Check for updates and alerts
                if processed % 10 == 0 or fraud_prob > alert_threshold:
                    clear_output(wait=True)
                    dashboard_html, fig = self.create_dashboard()
                    display(dashboard_html)
                    display(fig)
                
                if fraud_prob > alert_threshold:
                    self.send_alert(record, fraud_prob)
                    
                elapsed = time.time() - start_time
                remaining = max(0, interval - (elapsed % interval))
                time.sleep(remaining)
        
        except KeyboardInterrupt:
            print("Simulation stopped by user")
        
        # After the loop ends: Create DataFrame and print summaries
        fraud_log_df = pd.DataFrame(self.fraud_log)
        
        # 1. Print total simulation time (the "total simulated time" you asked for)
        total_sim_time = time.time() - start_time
        print(f"\nSimulation completed in {total_sim_time:.2f} seconds ({total_sim_time/60:.2f} minutes)")
        
        # 2. Print average inference time (per record, if logged)
        if 'inference_time_ms' in fraud_log_df.columns and not fraud_log_df.empty:
            avg_inference = fraud_log_df['inference_time_ms'].mean()
            print(f"Average inference time: {avg_inference:.2f} ms per record")
            print(f"Total records processed: {len(fraud_log_df)}")
            print(f"Overall fraud alert rate: {fraud_log_df['fraud_alert'].mean():.2%}")
        else:
            print("\nNo inference times were logged. Make sure timing is added in process_record().")
        
        return fraud_log_df



# Data Simulation

In [19]:
class MinistryFinancePayrollSimulator:
    """Simulator for generating realistic Ministry of Finance payroll data with fraud patterns"""
    
    def __init__(self, random_seed=42, duplicate_payment_rate=0.01, max_duplicate_payments=2):
        """Initialize the simulator with reproducible randomness"""
        self.fake = Faker()
        np.random.seed(random_seed)
        random.seed(random_seed)
        Faker.seed(random_seed)
        self.random_seed = random_seed
        
        # Set reference date for April 2025 salary
        self.reference_date = datetime(2025, 4, 1)
        self.duplicate_payment_rate = duplicate_payment_rate  # ~1% of employee-months
        self.max_duplicate_payments = max(1, max_duplicate_payments) 
        
        # Ministry of Finance specific data
        self.emp_departments = [
            'Finance', 'Budget', 'Treasury', 'Revenue', 'Economic Research', 
            'Debt Management', 'Investment', 'Procurement', 'Internal Audit', 
            'Legal Services', 'Information Technology', 'Human Resources', 
            'International Economics', 'Government Asset'
        ]
        
        self.emp_states = [
            'Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 
            'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 
            'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 
            'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 'Oyo', 
            'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara', 'FCT'
        ]
        
        self.emp_banks = [
            'Zenith', 'GTB', 'Access', 'First Bank', 'UBA', 'Sterling', 
            'Ecobank', 'Wema', 'Union', 'Globus'
        ]
        
        self.grade_levels = ['GL 08', 'GL 10', 'GL 12', 'GL 14', 'GL 15', 'GL 16-17']
        
        self.level_salary_mapping = {
            'GL 08': {
                'positions': ['Administrative Officer II', 'Accountant II', 'Budget Officer II', 
                             'Revenue Officer II', 'Procurement Officer II'],
                'salary_range': (90000, 120000),
                'retirement_age': 60
            },
            'GL 10': {
                'positions': ['Administrative Officer I', 'Accountant I', 'Budget Officer I', 
                             'Revenue Officer I', 'Procurement Officer I'],
                'salary_range': (120000, 150000),
                'retirement_age': 60
            },
            'GL 12': {
                'positions': ['Chief Administrative Officer', 'Principal Accountant', 
                             'Senior Budget Officer', 'Principal Revenue Officer'],
                'salary_range': (180000, 250000),
                'retirement_age': 60
            },
            'GL 14': {
                'positions': ['Assistant Director Finance', 'Assistant Director Budget', 
                             'Assistant Director Revenue', 'Assistant Director Audit'],
                'salary_range': (350000, 500000),
                'retirement_age': 60
            },
            'GL 15': {
                'positions': ['Deputy Director Finance', 'Deputy Director Budget', 
                             'Deputy Director Revenue', 'Deputy Director Economic Research'],
                'salary_range': (500000, 750000),
                'retirement_age': 60
            },
            'GL 16-17': {
                'positions': ['Director Finance', 'Director Budget', 'Director Treasury', 
                             'Director Revenue', 'Director-General'],
                'salary_range': (800000, 1500000),
                'retirement_age': 60  # Higher retirement age for top management
            }
        }
        
        # Department-specific overtime factors
        self.dept_overtime_factors = {
            'Finance': 1.2, 'Budget': 1.8, 'Treasury': 1.1, 'Revenue': 1.5,
            'Economic Research': 1.3, 'Debt Management': 1.1, 'Investment': 1.2,
            'Procurement': 1.6, 'Internal Audit': 1.4, 'Legal Services': 1.1,
            'Information Technology': 1.3, 'Human Resources': 1.1,
            'International Economics': 1.2, 'Government Asset': 1.1
        }
    
    def generate_10_digit_account(self):
        """Generate a 10-digit bank account number"""
        return str(random.randint(1000000000, 9999999999))
    
    def calculate_age(self, birth_date, reference_date=None):
        """Calculate age from birth date"""
        if reference_date is None:
            reference_date = self.reference_date
        return reference_date.year - birth_date.year - ((reference_date.month, reference_date.day) < (birth_date.month, birth_date.day))
    
    def simulate_employee_base(self, num_employees=1000):
        """Simulate base employee information for Ministry of Finance"""
        
        employees = []
        manager_ids = list(range(1, max(2, num_employees // 15 + 1)))  # Fewer managers in government
        
        for emp_id in range(1, num_employees + 1):
            # Assign grade level with distribution (more lower grades)
            grade_weights = [0.25, 0.25, 0.20, 0.15, 0.10, 0.05]  # GL08, GL10, GL12, GL14, GL15, GL16-17
            grade_level = random.choices(self.grade_levels, weights=grade_weights, k=1)[0]
            
            # Get position based on grade level
            position = random.choice(self.level_salary_mapping[grade_level]['positions'])
            
            # Get BASE salary based on grade level (this is the correct salary)
            salary_min, salary_max = self.level_salary_mapping[grade_level]['salary_range']
            base_salary = random.randint(salary_min, salary_max)

            # Assign gender (70% Male, 30% Female)
            gender_weights = [0.7, 0.3]
            gender = random.choices(["Male", "Female"], weights=gender_weights, k=1)[0]

            # Assign department (some departments have more staff)
            dept_weights = [0.12, 0.10, 0.08, 0.12, 0.06, 0.05, 0.05, 0.08, 0.06, 0.04, 0.07, 0.05, 0.04, 0.08]
            department = random.choices(self.emp_departments, weights=dept_weights, k=1)[0]
            
            # Generate birth date (ages between 25-65)
            birth_year = random.randint(self.reference_date.year - 65, self.reference_date.year - 25)
            birth_date = datetime(birth_year, random.randint(1, 12), random.randint(1, 28))
            age = self.calculate_age(birth_date, self.reference_date)
            
            # Government-specific details
            hire_date = self.fake.date_between(start_date='-30y', end_date='-1y')  # Longer tenure
            state_of_origin = random.choice(self.emp_states)
            bank_name = random.choice(self.emp_banks)
            bank_account = self.generate_10_digit_account()
            
            # Determine employment status
            retirement_age = self.level_salary_mapping[grade_level]['retirement_age']
            status = self._determine_employment_status(age, retirement_age, hire_date)
            
            employees.append({
                'EmployeeID': f"MOF{emp_id:04d}",
                'Name': self.fake.name(),
                'Gender': gender,
                'GradeLevel': grade_level,
                'Position': position,
                'Department': department,
                'BaseSalary': base_salary,  # Correct salary for grade level
                'HireDate': hire_date,
                'BirthDate': birth_date,
                'Age': age,
                'EmploymentStatus': status,
                'StateOfOrigin': state_of_origin,
                'BankName': bank_name,
                'BankAccount': bank_account,
                'ManagerID': random.choice(manager_ids) if emp_id > 15 else 'N/A'
            })
        
        return pd.DataFrame(employees)
    
    def _determine_employment_status(self, age, retirement_age, hire_date):
        """Determine employment status based on age and other factors"""
        
        # Base probabilities
        if age >= retirement_age + 1:  
            return 'Inactive' if random.random() < 0.8 else 'Active'
        elif age >= retirement_age:  # At or slightly above retirement age
            return 'Inactive' if random.random() < 0.4 else 'Active'
        elif age >= retirement_age - 5:  # Nearing retirement
            return 'Active'
        else:  # Younger employees
            return 'Inactive' if random.random() < 0.05 else 'Active'
    
    def introduce_salary_fraud(self, employee_df):
        """Introduce salary fraud by modifying paid salaries above grade levels"""
        
        df = employee_df.copy()
        
        # Add PaidSalary column (initially same as BaseSalary)
        df['PaidSalary'] = df['BaseSalary']
        
        for idx, emp in df.iterrows():
            salary_min, salary_max = self.level_salary_mapping[emp['GradeLevel']]['salary_range']
            
            # 5% chance of salary fraud for each employee
            if random.random() < 0.05:
                # Inflate salary above grade level
                inflation_factor = random.uniform(1.1, 2.0)  # 10% to 100% over grade limit
                inflated_salary = min(salary_max * inflation_factor, salary_max * 2.5)  # Cap at 2.5x
                df.at[idx, 'PaidSalary'] = int(inflated_salary)
        
        return df
    
    def simulate_payroll_transactions(self, employee_df, months=12, base_fraud_rate=0.04, duplicate_fraud_rate=0.01):
        """Simulate monthly payroll transactions with embedded fraud patterns"""
        
        payroll_records = []
        
        # Generate periods including April 2025
        periods = []
        for month in range(months):
            period_date = (self.reference_date - timedelta(days=30*month)).strftime('%Y-%m-01')
            periods.append(period_date)
        
        # Reverse to get chronological order
        periods = periods[::-1]
        
        for period_date in periods:
            current_period_date = datetime.strptime(period_date, '%Y-%m-%d')
            
            for _, emp in employee_df.iterrows():
                # Calculate current age for this period
                current_age = self.calculate_age(emp['BirthDate'], current_period_date)
                
                # Base values with department-specific patterns
                regular_hours = 160 if emp['EmploymentStatus'] == 'Active' else 0
                
                # Overtime based on department and grade level (only for active employees)
                if emp['EmploymentStatus'] == 'Active':
                    overtime_factor = self.dept_overtime_factors[emp['Department']]
                    base_overtime = max(0, int(np.random.normal(8, 6))) * overtime_factor
                    overtime_hours = int(base_overtime)
                else:
                    overtime_hours = 0
                
                # Expense claims (only for active employees)
                if emp['EmploymentStatus'] == 'Active':
                    base_expense = 200 if emp['GradeLevel'] in ['GL 08', 'GL 10'] else 400
                    expense_amount = max(0, int(np.random.normal(base_expense, 150)))
                else:
                    expense_amount = 0
                
                # Allowances (government specific)
                housing_allowance = emp['BaseSalary'] * 0.15
                transport_allowance = emp['BaseSalary'] * 0.10
                if emp['GradeLevel'] in ['GL 14', 'GL 15', 'GL 16-17']:
                    transport_allowance *= 1.5
                
                # Get the actual paid salary
                paid_salary = emp['PaidSalary']
                
                # Initialize as non-fraudulent
                is_fraudulent = 0
                fraud_type = 'None'
                fraud_severity = 0.0
                salary_violation = 0.0
                
                # Get salary range for this grade level
                salary_min, salary_max = self.level_salary_mapping[emp['GradeLevel']]['salary_range']
                retirement_age = self.level_salary_mapping[emp['GradeLevel']]['retirement_age']
                
                # AUTOMATIC FRAUD FLAGS
                
                # Flag 1: Salary above grade level range
                if paid_salary > salary_max and random.random() < 0.8:
                    is_fraudulent = 1
                    fraud_type = 'salary_above_grade'
                    fraud_severity = (paid_salary - salary_max) / salary_max  # Percentage over
                    salary_violation = paid_salary - salary_max
                
                # Flag 2: Inactive employees receiving payments (Ghost workers)
                elif (emp['EmploymentStatus'] == 'Inactive' and 
                      (paid_salary > 0 or expense_amount > 0 or overtime_hours > 0) and
                      random.random() < 0.9):
                    is_fraudulent = 1
                    fraud_type = 'ghost_employee'
                    fraud_severity = random.uniform(1.0, 2.0)
                    # Zero out all payments for ghost workers
                    regular_hours = 0
                    overtime_hours = 0
                    expense_amount = 0
                
                # Flag 3: Employees over retirement age still receiving salary
                elif (current_age > retirement_age and 
                      emp['EmploymentStatus'] == 'Active' and 
                      random.random() < 0.7):
                    is_fraudulent = 1
                    fraud_type = 'overage_employee'
                    fraud_severity = random.uniform(0.8, 1.5)
                    # Reduce activity for overage employees
                    regular_hours *= random.uniform(0.7, 0.9)
                    overtime_hours *= random.uniform(0.5, 0.8)
                
                # Regular fraud patterns for active employees
                elif (emp['EmploymentStatus'] == 'Active' and 
                      current_age <= retirement_age and 
                      random.random() < base_fraud_rate):
                    is_fraudulent = 1
                    fraud_type, fraud_severity = self._simulate_fraud_pattern(emp)
                    
                    # Apply fraud modifications
                    if fraud_type == 'overtime_inflation':
                        overtime_hours = int(overtime_hours * random.uniform(2.0, 4.0))
                    elif fraud_type == 'expense_inflation':
                        expense_amount = int(expense_amount * random.uniform(3.0, 8.0))
                    elif fraud_type == 'allowance_inflation':
                        housing_allowance *= random.uniform(1.5, 2.5)
                        transport_allowance *= random.uniform(1.5, 2.5)
                
                base_record = {
                    'Period': period_date,
                    'EmployeeID': emp['EmployeeID'],
                    'PaidSalary': paid_salary,
                    'RegularHours': regular_hours,
                    'OvertimeHours': overtime_hours,
                    'ExpenseAmount': expense_amount,
                    'HousingAllowance': housing_allowance,
                    'TransportAllowance': transport_allowance,
                    'IsFraudulent': is_fraudulent,
                    'FraudType': fraud_type,
                    'FraudSeverity': fraud_severity,
                    'SalaryViolationAmount': salary_violation,
                    'CurrentAge': current_age,
                    # NEW tracking fields
                    'TransactionID': str(uuid.uuid4()),
                    'PaymentSequence': 1,                 # first payment this month
                    'IsDuplicatePayment': 0,              # base record is not a duplicate
                    'DuplicateGroupID': None              # no group for the legit record
                }
                payroll_records.append(base_record)
                # Fraud injection: duplicate payment (same salary again)
                if np.random.rand() < duplicate_fraud_rate:
                    duplicate_record = base_record.copy()
                    duplicate_record["FraudType"] = "DuplicatePayment"
                    duplicate_record["IsFraudulent"] = 1
                    duplicate_record["IsDuplicatePayment"] = 1
                    payroll_records.append(duplicate_record)
        # Add PaidSalary to the payroll dataframe for easier merging
        #payroll_records['PaidSalary'] = payroll_records['PaidSalary']  # This ensures it's in the payroll df
        payroll_df = pd.DataFrame(payroll_records)
        return payroll_df
    
    def _simulate_fraud_pattern(self, employee):
        """Simulate specific fraud patterns for active employees"""
        
        fraud_patterns = [
            ('overtime_inflation', 0.40),
            ('expense_inflation', 0.35),
            ('allowance_inflation', 0.25)
        ]
        
        if employee['Department'] in ['Procurement', 'Budget']:
            fraud_patterns = [
                ('overtime_inflation', 0.30),
                ('expense_inflation', 0.50),
                ('allowance_inflation', 0.20)
            ]
        
        patterns, weights = zip(*fraud_patterns)
        chosen_pattern = random.choices(patterns, weights=weights, k=1)[0]
        severity = random.uniform(0.5, 2.0)
        
        return chosen_pattern, severity
    
    def generate_complete_dataset(self, num_employees=800, months=24, fraud_rate=0.04):
        """Generate a complete simulated Ministry of Finance payroll dataset"""
        
        print("Simulating Ministry of Finance employee base data...")
        employees = self.simulate_employee_base(num_employees)
        
        print("Introducing salary fraud...")
        employees = self.introduce_salary_fraud(employees)
        
        print("Simulating payroll transactions...")
        payroll = self.simulate_payroll_transactions(employees, months, fraud_rate)
        
        # Merge with employee data - include all necessary columns
        merge_columns = ['EmployeeID', 'Name', 'Gender','GradeLevel', 'Position', 'Department', 
                        'HireDate', 'BirthDate', 'Age', 'EmploymentStatus', 'StateOfOrigin',
                        'BankName', 'BankAccount', 'ManagerID', 'BaseSalary']
        
        merged_df = pd.merge(payroll, employees[merge_columns], on='EmployeeID', how='left')


        
        # Calculate financial metrics
        merged_df['HourlyRate'] = merged_df['PaidSalary'] / 2080
        merged_df['RegularPay'] = merged_df['RegularHours'] * merged_df['HourlyRate']
        merged_df['OvertimePay'] = merged_df['OvertimeHours'] * merged_df['HourlyRate'] * 1.5
        merged_df['TotalAllowances'] = merged_df['HousingAllowance'] + merged_df['TransportAllowance']
        merged_df['GrossSalary'] = merged_df['PaidSalary'] + merged_df['TotalAllowances']
        merged_df['TotalCompensation'] = merged_df['GrossSalary'] + merged_df['OvertimePay'] + merged_df['ExpenseAmount']
        
        # Calculate salary compliance
        def get_salary_range(grade_level):
            return self.level_salary_mapping[grade_level]['salary_range']
        
        merged_df['SalaryMin'] = merged_df['GradeLevel'].apply(lambda x: get_salary_range(x)[0])
        merged_df['SalaryMax'] = merged_df['GradeLevel'].apply(lambda x: get_salary_range(x)[1])
        merged_df['IsSalaryAboveGrade'] = merged_df['PaidSalary'] > merged_df['SalaryMax']
        merged_df['SalaryDeviation'] = merged_df['PaidSalary'] - merged_df['SalaryMax']
        merged_df['SalaryDeviationPct'] = (merged_df['SalaryDeviation'] / merged_df['SalaryMax']) * 100
        
        # Add retirement age info
        merged_df['RetirementAge'] = merged_df['GradeLevel'].map(
            lambda x: self.level_salary_mapping[x]['retirement_age']
        )
        merged_df['IsOverRetirementAge'] = merged_df['CurrentAge'] > merged_df['RetirementAge']
        
        # Final column ordering
        columns_order = [
            'Period', 'EmployeeID', 'Name', 'Gender', 'GradeLevel', 'Position', 'Department', 
            'HireDate', 'BirthDate', 'Age', 'CurrentAge', 'RetirementAge', 'IsOverRetirementAge',
            'EmploymentStatus', 'StateOfOrigin', 'BankName', 'BankAccount', 'ManagerID',
            'BaseSalary', 'PaidSalary', 'SalaryMin', 'SalaryMax', 'IsSalaryAboveGrade',
            'SalaryDeviation', 'SalaryDeviationPct', 'SalaryViolationAmount',
            'HourlyRate', 'RegularHours', 'RegularPay', 'OvertimeHours', 'OvertimePay', 
            'HousingAllowance', 'TransportAllowance', 'TotalAllowances', 'ExpenseAmount', 
            'GrossSalary', 'TotalCompensation', 'IsFraudulent', 'FraudType', 'FraudSeverity',
            'TransactionID', 'PaymentSequence', 'IsDuplicatePayment', 'DuplicateGroupID'
        ]
        
        return merged_df[columns_order]
    
    def generate_statistics(self, simulated_data):
        """Generate comprehensive statistics from the simulated data"""
        
        print(f"\nSimulation completed successfully!")
        print(f"Generated {len(simulated_data)} records")
        print(f"Overall fraud rate: {simulated_data['IsFraudulent'].mean()*100:.2f}%")
        
        print("\nFraud type distribution:")
        fraud_stats = simulated_data[simulated_data['IsFraudulent'] == 1]['FraudType'].value_counts()
        for fraud_type, count in fraud_stats.items():
            percentage = (count / len(simulated_data[simulated_data['IsFraudulent'] == 1])) * 100
            print(f"  {fraud_type}: {count} cases ({percentage:.1f}%)")
        
        print("\nSalary Fraud Analysis:")
        salary_fraud = simulated_data[simulated_data['IsSalaryAboveGrade'] == True]
        print(f"  Employees with salary above grade level: {len(salary_fraud['EmployeeID'].unique())}")
        if len(salary_fraud) > 0:
            print(f"  Average salary deviation: ₦{salary_fraud['SalaryDeviation'].mean():,.0f}")
            print(f"  Maximum salary deviation: ₦{salary_fraud['SalaryDeviation'].max():,.0f}")
        else:
            print("  No salary fraud detected")
        
        print("\nAge-Based Fraud Analysis:")
        overage_employees = simulated_data[simulated_data['IsOverRetirementAge'] == True]
        print(f"  Employees over retirement age: {len(overage_employees['EmployeeID'].unique())}")
        if len(overage_employees) > 0:
            print(f"  Fraud rate among overage employees: {overage_employees['IsFraudulent'].mean()*100:.1f}%")
        else:
            print("  No employees over retirement age")
        
        # Show April 2025 data specifically
        april_2025_data = simulated_data[simulated_data['Period'] == '2025-04-01']
        print(f"\nApril 2025 Payroll Analysis:")
        print(f"  Records in April 2025: {len(april_2025_data)}")
        print(f"  Fraud rate in April 2025: {april_2025_data['IsFraudulent'].mean()*100:.1f}%")
        
        # Show sample with salary fraud details
        print("\nSample data (showing salary fraud details):")
        sample_cols = ['EmployeeID', 'Name', 'Gender', 'GradeLevel', 'BaseSalary', 'PaidSalary', 
                      'SalaryMax', 'IsSalaryAboveGrade', 'CurrentAge', 'IsOverRetirementAge', 
                      'IsFraudulent', 'FraudType']
        sample_data = simulated_data[sample_cols].drop_duplicates().head(8)
        print(sample_data.to_string(index=False))

def main():
    """Main function to generate and save simulated payroll data"""
    
    # Initialize simulator
    simulator = MinistryFinancePayrollSimulator(random_seed=42)
    
    # Generate simulated data
    print("Starting Ministry of Finance payroll data simulation...")
    simulated_data = simulator.generate_complete_dataset(
        num_employees=1000, 
        months=12,           
        fraud_rate=0.03
    )
    
    # Save to CSV
    output_file = "ministry_of_finance_payroll_data.csv"
    simulated_data.to_csv(output_file, index=False)
    
    # Generate statistics
    simulator.generate_statistics(simulated_data)
    
    print(f"\nSimulated data saved to: {output_file}")

if __name__ == "__main__":
   main()


Starting Ministry of Finance payroll data simulation...
Simulating Ministry of Finance employee base data...
Introducing salary fraud...
Simulating payroll transactions...

Simulation completed successfully!
Generated 12110 records
Overall fraud rate: 17.42%

Fraud type distribution:
  ghost_employee: 1117 cases (52.9%)
  salary_above_grade: 418 cases (19.8%)
  overage_employee: 136 cases (6.4%)
  overtime_inflation: 127 cases (6.0%)
  expense_inflation: 116 cases (5.5%)
  DuplicatePayment: 110 cases (5.2%)
  allowance_inflation: 86 cases (4.1%)

Salary Fraud Analysis:
  Employees with salary above grade level: 43
  Average salary deviation: ₦209,121
  Maximum salary deviation: ₦1,237,358

Age-Based Fraud Analysis:
  Employees over retirement age: 81
  Fraud rate among overage employees: 86.1%

April 2025 Payroll Analysis:
  Records in April 2025: 1012
  Fraud rate in April 2025: 17.6%

Sample data (showing salary fraud details):
EmployeeID            Name Gender GradeLevel  BaseSalary

# Main Execution

In [ ]:
def main():
    set_seeds(42)  # Consistent seed everywhere
    
    # ==================== DATA PREPARATION ========================
    try:
        raw_df = pd.read_csv('C://Users/AsogwaKenneth/ministry_of_finance_payroll_data.csv')
        #raw_df = MinistryFinancePayrollSimulator(random_seed=42)
    except FileNotFoundError:
        print("Error: The CSV file was not found at the specified path.")
        return
    
    # Initialize simulator for sensitivity analysis
    simulator = MinistryFinancePayrollSimulator(random_seed=42)
    
    label = 'IsFraudulent'
    numeric_features = [
        'PaidSalary', 'RegularHours', 'OvertimeHours', 'ExpenseAmount',
        'HousingAllowance', 'TransportAllowance', 'Age', 'CurrentAge',
        'RetirementAge', 'SalaryDeviation', 'SalaryDeviationPct'
    ]
    categorical_features = ['Department', 'Gender', 'GradeLevel', 'Position', 'StateOfOrigin']
    
    data = pd.get_dummies(raw_df[numeric_features + categorical_features], columns=categorical_features, dtype=float)
    model_features = data.columns.tolist()
    joblib.dump(model_features, 'model_features.pkl')
    print(f"Saved {len(model_features)} feature columns to 'model_features.pkl'")
    
    X = data.values
    y = raw_df[label].values
    
    scaler = MinMaxScaler()
    scaler.fit(data)
    joblib.dump(scaler, 'scaler.pkl')
    print("Saved scaler to 'scaler.pkl'")
    
    X_scaled = scaler.transform(data)
    timesteps = 1
    input_shape = (timesteps, X_scaled.shape[1])
    X_reshaped = X_scaled.reshape(-1, *input_shape)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_reshaped, y, test_size=0.2, random_state=52, stratify=y
    )
    
    # ========== MODEL TRAINING ==========
    model = None
    best_hps_found = None
    cv_scores = None  # Default - only set if CV runs
    
    option = "Hyperparameter Tuning"  # Change to "Cross-Validation" if preferred
    
    if option == "Hyperparameter Tuning":
        print("\n--- Initiating Hyperparameter Tuning ---")
        try:
            tuned_model, best_hps_found = optimize_model(X_train, y_train, input_shape)
            model = tuned_model
            best_hps = best_hps_found.values
        except Exception as e:
            print(f"Error during tuning: {str(e)}")
            option = "Cross-Validation"
            best_hps = None
    
    if option == "Cross-Validation":
        print("\n--- Initiating Cross-Validation ---")
        try:
            model, cv_scores = cross_validate_model(X_train, y_train, input_shape, best_hps=best_hps_found)
        except Exception as e:
            print(f"Error during cross-validation: {str(e)}")
            return
    
    # Baselines
    print("\n--- Training Baseline Models ---")
    baseline_results = train_baselines(X_train, y_train, input_shape, cv_scores=cv_scores)
    print("Baseline Results:", baseline_results)

    baseline_df = pd.DataFrame.from_dict(baseline_results, orient='index')
    baseline_df = baseline_df.rename_axis('Model').reset_index()  # Makes "Model" a column
    baseline_df.to_csv('baseline_results.csv', index=False)
    
    #baseline_results.to_csv('baseline_results.csv', index=False)
    
    # Save hyperparameters for reproducibility
    if 'best_hps' in locals() and best_hps is not None:
        config = {
            'hyperparameters': best_hps,
            'input_shape': input_shape,
            'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
            'option_used': option
        }
        with open('model_hyperparameters.json', 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=4)
        print("Hyperparameters saved to 'model_hyperparameters.json'")
    else:
        default_config = {
            'note': 'No hyperparameter tuning - using defaults',
            'input_shape': input_shape,
            'timestamp': pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        with open('model_hyperparameters.json', 'w', encoding='utf-8') as f:
            json.dump(default_config, f, indent=4)
        print("Default configuration saved to 'model_hyperparameters.json'")
    
    # Plot CNN architecture
    if model is not None:
        try:
            tf.keras.utils.plot_model(
                model,
                to_file='cnn_model_architecture.png',
                show_shapes=True,
                show_layer_names=True
            )
            print("CNN model architecture plot saved as 'cnn_model_architecture.png'")
        except Exception as e:
            print(f"Plotting error: {e}")
    
    if model:
        # ========== FINAL TRAINING ==========
        print("\n--- Training Final Model ---")
        smote = SMOTE(sampling_strategy=0.5, random_state=52)
        X_res, y_res = smote.fit_resample(
            X_train.reshape(X_train.shape[0], -1),
            y_train
        )
        X_res = X_res.reshape(-1, *input_shape)
        history = model.fit(
            X_res, y_res,
            epochs=50,
            batch_size=64,
            class_weight={0: 1, 1: 10},
            validation_split=0.1,
            callbacks=[
                EarlyStopping(monitor='val_pr_auc', patience=5, mode='max', restore_best_weights=True)
            ],
            verbose=1
        )
        
        # ========== THRESHOLD & BINARY METRICS ==========
        print("\n--- Finding Optimal Threshold ---")
        def find_best_threshold(y_true, y_pred_prob, method="f1"):
            precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_prob)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
            if method == "f1":
                best_idx = np.argmax(f1_scores)
                best_threshold = thresholds[best_idx]
                print(f"\nBest Threshold (F1): {best_threshold:.3f}, F1: {f1_scores[best_idx]:.3f}")
                return best_threshold
        
        y_pred_proba = model.predict(X_test)
        y_pred_proba_fraud = y_pred_proba.flatten()
        optimal_threshold = find_best_threshold(y_test, y_pred_proba_fraud, method="f1")
        
        y_test_fraud = y_test
        y_pred_class_fraud = (y_pred_proba_fraud >= optimal_threshold).astype(int)

        # ================== SAVE OPTIMAL THRESHOLD TO CSV ==================
        threshold_df = pd.DataFrame({
            'Metric': ['Optimal Threshold (F1 method)'],
            'Value': [optimal_threshold]
            #'F1_Score_at_Threshold': [f1_score_fraud]   # optional but useful
        })
        threshold_df.to_csv('optimal_threshold.csv', index=False)
        
        precision_score_fraud = precision_score(y_test_fraud, y_pred_class_fraud, pos_label=1, zero_division=0)
        recall_score_fraud = recall_score(y_test_fraud, y_pred_class_fraud, pos_label=1, zero_division=0)
        f1_score_fraud = f1_score(y_test_fraud, y_pred_class_fraud, pos_label=1, zero_division=0)
        accuracy_score_total = (y_test_fraud == y_pred_class_fraud).mean()
        pr_auc = average_precision_score(y_test_fraud, y_pred_proba_fraud)
        
        
        print("\n--- Ablation: Training without SMOTE ---")
        model_no_smote = clone_model(model)
        model_no_smote.set_weights(model.get_weights())
        
        lr_value = float(model.optimizer.learning_rate)
        model_no_smote.compile(
            optimizer=model.optimizer.__class__(learning_rate=lr_value),
            loss='binary_crossentropy',
            metrics=[tf.keras.metrics.AUC(name='pr_auc', curve='PR')]
        )
        
        history_no_smote = model_no_smote.fit(
            X_train, y_train,
            epochs=50,
            batch_size=64,
            class_weight={0: 1, 1: 10},
            validation_split=0.1,
            callbacks=[EarlyStopping(monitor='val_pr_auc', patience=5, mode='max', restore_best_weights=True)],
            verbose=1
        )
        
        y_pred_no_smote = model_no_smote.predict(X_test)
        y_pred_class_no_smote = (y_pred_no_smote.flatten() >= optimal_threshold).astype(int)
        
        precision_no_smote = precision_score(y_test, y_pred_class_no_smote, pos_label=1, zero_division=0)
        recall_no_smote = recall_score(y_test, y_pred_class_no_smote, pos_label=1, zero_division=0)
        f1_no_smote = f1_score(y_test, y_pred_class_no_smote, pos_label=1, zero_division=0)
        
        print(f"Without SMOTE → Precision: {precision_no_smote:.4f}, Recall: {recall_no_smote:.4f}, F1: {f1_no_smote:.4f}")
        print(f"With SMOTE → Precision: {precision_score_fraud:.4f}, Recall: {recall_score_fraud:.4f}, F1: {f1_score_fraud:.4f}")
        
        ablation_df = pd.DataFrame({
            'Method': ['With SMOTE', 'Without SMOTE'],
            'Precision': [precision_score_fraud, precision_no_smote],
            'Recall': [recall_score_fraud, recall_no_smote],
            'F1': [f1_score_fraud, f1_no_smote]
        })
        ablation_df.to_csv('ablation_results.csv', index=False)


    
    # Save model and summary
    model.save('best_model.h5')
    with open('model_summary.txt', 'w', encoding='utf-8') as f:
        model.summary(print_fn=lambda x: f.write(x + '\n'))
    logging.info("Model summary saved to model_summary.txt")
    
    # ========== EVALUATION ==========
    print("\n--- Evaluating Final Model on Test Set ---")
    
    # Bootstrapped CIs
    print("\n--- Bootstrapped Confidence Intervals ---")
    def bootstrap_ci(y_true, y_pred, metric, n_bootstrap=1000):
        scores = []
        n = len(y_true)
        for _ in range(n_bootstrap):
            idx = np.random.choice(n, n, replace=True)
            scores.append(metric(y_true[idx], y_pred[idx]))
        low, high = np.percentile(scores, [2.5, 97.5])
        return np.mean(scores), low, high
    
    f1_mean, f1_low, f1_high = bootstrap_ci(y_test_fraud, y_pred_class_fraud, f1_score)
    prec_mean, prec_low, prec_high = bootstrap_ci(y_test_fraud, y_pred_class_fraud, precision_score)
    rec_mean, rec_low, rec_high = bootstrap_ci(y_test_fraud, y_pred_class_fraud, recall_score)
    acc_mean, acc_low, acc_high = bootstrap_ci(y_test_fraud, y_pred_class_fraud, accuracy_score)
    pr_mean, pr_low, pr_high = bootstrap_ci(y_test_fraud, y_pred_proba_fraud, average_precision_score)

    
    print(f"F1-score 95% CI: {f1_mean:.3f} [{f1_low:.3f}, {f1_high:.3f}]")
    print(f"Precision 95% CI: {prec_mean:.3f} [{prec_low:.3f}, {prec_high:.3f}]")
    print(f"Recall 95% CI: {rec_mean:.3f} [{rec_low:.3f}, {rec_high:.3f}]")
    print(f"Accuracy 95% CI: {acc_mean:.3f} [{acc_low:.3f}, {acc_high:.3f}]")
    print(f"PR-AUC 95% CI: {pr_mean:.3f} [{pr_low:.3f}, {pr_high:.3f}]")

    # ================== SAVE TO CSV ==================
    ci_data = {
        'Metric': ['F1-score', 'Precision', 'Recall', 'Accuracy'],
        'Mean': [f1_mean, prec_mean, rec_mean, acc_mean],
        'Lower_95_CI': [f1_low, prec_low, rec_low, acc_low],
        'Upper_95_CI': [f1_high, prec_high, rec_high, acc_high]
    }

    ci_df = pd.DataFrame(ci_data)
    ci_df.to_csv('bootstrapped_confidence_intervals.csv', index=False)
    print("Bootstrapped confidence intervals saved to 'bootstrapped_confidence_intervals.csv'")

    # Sensitivity analysis
    print("\n--- Running Sensitivity Analysis on Fraud Rate ---")
    fraud_rates = [0.02, 0.03, 0.04, 0.05]
    sensitivity_results = {}
    for rate in fraud_rates:
        print(f"Testing fraud_rate = {rate}")
        simulated_data = simulator.generate_complete_dataset(
            num_employees=1000, months=12, fraud_rate=rate
        )
        
        sens_data = pd.get_dummies(simulated_data[numeric_features + categorical_features],
                                   columns=categorical_features, dtype=float)
        
        for col in model_features:
            if col not in sens_data.columns:
                sens_data[col] = 0
        sens_data = sens_data[model_features]
        
        X_sens = sens_data.values
        y_sens = simulated_data['IsFraudulent'].values
        
        X_sens_scaled = scaler.transform(X_sens)
        X_sens_reshaped = X_sens_scaled.reshape(-1, *input_shape)
        
        y_pred_sens_prob = model.predict(X_sens_reshaped)
        y_pred_sens = (y_pred_sens_prob.flatten() >= optimal_threshold).astype(int)
        
        f1_sens = f1_score(y_sens, y_pred_sens, zero_division=0)
        sensitivity_results[rate] = f1_sens
        print(f"F1 at fraud_rate {rate}: {f1_sens:.4f}")
    
    pd.DataFrame.from_dict(sensitivity_results, orient='index', columns=['F1']).to_csv('sensitivity_analysis.csv')

# ========== SHAP FEATURE IMPORTANCE ==========
    print("\n--- Computing SHAP Feature Importance ---")
    
    # 1. Prepare subsets for explanation
    # DeepExplainer needs a representative background (100 samples is standard)
    background_3d = X_train[:100]  
    test_subset_3d = X_test[:200]  
    
    # 2. Initialize Explainer
    # Use DeepExplainer for Keras/TF models
    explainer = shap.DeepExplainer(model, background_3d)
    
    # 3. Compute SHAP values
    # For binary classification (sigmoid), shap_values is a list containing one array
    shap_values = explainer.shap_values(test_subset_3d)
    shap_values_plot = shap_values[0] if isinstance(shap_values, list) else shap_values
    
    # 4. Flatten for 2D Visualization
    # summary_plot and bar_plot expect (N_samples, N_features)
    shap_values_flat = shap_values_plot.reshape(shap_values_plot.shape[0], -1)
    test_subset_flat = test_subset_3d.reshape(test_subset_3d.shape[0], -1)
    
    # --- Visualization A: SHAP Summary (Beeswarm) ---
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_flat, 
        test_subset_flat, 
        feature_names=model_features, 
        show=False, 
        plot_type="dot"
    )
    plt.title("SHAP Feature Impact on Fraud Prediction (Beeswarm)", fontsize=14, pad=20)
    plt.savefig('shap_summary_beeswarm.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- Visualization B: Global Feature Importance (Bar) ---
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_flat, 
        test_subset_flat, 
        feature_names=model_features, 
        show=False, 
        plot_type="bar"
    )
    plt.title("Global Feature Importance (Average |SHAP|)", fontsize=14, pad=20)
    plt.savefig('shap_feature_importance_bar.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 5. Export Importance Data to CSV for Usability
    importance_df = pd.DataFrame({/
        'Feature': model_features,
        'Mean_Abs_SHAP': np.abs(shap_values_flat).mean(axis=0)
    }).sort_values(by='Mean_Abs_SHAP', ascending=False)
    
    importance_df.to_csv('feature_importance_shap.csv', index=False)
    print("SHAP analysis complete. Plots and CSV importance saved.")
    
    # Real-time dashboard
    print("\n--- Starting Real-Time Fraud Detection Dashboard ---")
    best_model = tf.keras.models.load_model('best_model.h5')
    scaler = joblib.load('scaler.pkl')
    feature_cols = joblib.load('model_features.pkl')
    
    fraud_system = FraudDetectionSystem(
        model=best_model,
        df=raw_df,
        scaler=scaler,
        categorical_cols=categorical_features,
        feature_cols=feature_cols,
        seed=42
    )
    
    simulation_config = {
        'duration': 60,
        'interval': 0.1,
        'alert_threshold': optimal_threshold
    }
    
    print(f"Starting simulation for {simulation_config['duration']} seconds...")
    try:
        fraud_log_df = fraud_system.run_simulation(
            duration=simulation_config['duration'],
            interval=simulation_config['interval'],
            alert_threshold=simulation_config['alert_threshold']
        )
        
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        log_filename = f'fraud_detection_logs_{timestamp}.csv'
        fraud_log_df.to_csv(log_filename, index=False)
        print(f"Saved logs to '{log_filename}'")
        
        viz_filename = f'fraud_analysis_{timestamp}.html'
        if not fraud_log_df.empty:
            fig = px.line(fraud_log_df, x='timestamp', y='fraud_probability',
                          color='Department',
                          title=f'Fraud Probability Timeline | {len(fraud_log_df)} Records',
                          hover_data=['EmployeeID', 'Position', 'GradeLevel', 'PaidSalary'])
            fig.write_html(viz_filename)
            print(f"Saved visualization to '{viz_filename}'")
        
        # Final summaries
        print("\n=== Real-Time Simulation Results ===")
        print(f"Total Records processed: {len(fraud_log_df):,}")
        fraud_cases = fraud_log_df[fraud_log_df['fraud_probability'] > simulation_config['alert_threshold']]
        print(f"High-probability fraud cases detected: {len(fraud_cases):,}")
        
               # ==================== FINAL EVALUATION METRICS & PLOTS ====================
        print("\n=== Final Evaluation Metrics & Plots ===")

        # Confusion Matrix Heatmap
        cm_fraud = confusion_matrix(y_test_fraud, y_pred_class_fraud, labels=[0, 1])
        cm_fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm_fraud, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Predicted Non-Fraud', 'Predicted Fraud'],
                    yticklabels=['Actual Non-Fraud', 'Actual Fraud'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title('Confusion Matrix (Test Set)')
        plt.tight_layout()
        plt.savefig('confusion_matrix_final.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(cm_fig)

        # Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(y_test_fraud, y_pred_proba_fraud)
        pr_fig, ax_pr = plt.subplots(figsize=(8, 6))
        pr_display = PrecisionRecallDisplay(precision=precision, recall=recall, estimator_name="CNN Model")
        pr_display.plot(ax=ax_pr)
        ax_pr.set_title('Precision-Recall Curve (Test Set)')
        ax_pr.grid(True)
        plt.tight_layout()
        plt.savefig('precision_recall_curve_final.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(pr_fig)

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_test_fraud, y_pred_proba_fraud)
        roc_auc_score = roc_auc(fpr, tpr)
        roc_fig, ax_roc = plt.subplots(figsize=(8, 6))
        roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc_score, estimator_name="CNN Model")
        roc_display.plot(ax=ax_roc)
        ax_roc.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Chance', alpha=0.8)
        ax_roc.set_title('ROC Curve (Test Set)')
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate')
        ax_roc.legend(loc="lower right")
        ax_roc.grid(True)
        plt.tight_layout()
        plt.savefig('roc_curve_final.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(roc_fig)

        # Caliberation Curve
        # Ensure probabilities are 1D
        y_pred_proba_fraud = y_pred_proba_fraud.ravel()
        
        # Compute calibration curve
        prob_true, prob_pred = calibration_curve(y_test_fraud, y_pred_proba_fraud, n_bins=10)
        
        # Plot
        calib_fig, ax_calib = plt.subplots(figsize=(8, 6))
        
        ax_calib.plot(prob_pred, prob_true, marker='o', label='CNN Model')
        ax_calib.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Perfect Calibration')
        
        ax_calib.set_title('Calibration Curve')
        ax_calib.set_xlabel('Predicted Probability')
        ax_calib.set_ylabel('True Probability')
        ax_calib.legend(loc="upper left")
        ax_calib.grid(True)
        
        plt.tight_layout()
        plt.savefig('calibration_curve_final.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(calib_fig)

        # Cost Sensitivity Analysis
        # Define misclassification costs
        C_FN = 10   # Missing fraud (very expensive)
        C_FP = 1    # False alarm (less expensive)
        thresholds = np.linspace(0, 1, 100)
        costs = []
        
        for t in thresholds:
            y_pred_temp = (y_pred_proba_fraud >= t).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_test_fraud, y_pred_temp).ravel()
            cost = (C_FN * fn) + (C_FP * fp)
            costs.append(cost)
        
        best_threshold = thresholds[np.argmin(costs)]
        
        print(f"Optimal Threshold: {best_threshold:.3f}")

        # Plot
        cost_fig, ax_cost = plt.subplots(figsize=(8, 6))
        
        ax_cost.plot(thresholds, costs)
        ax_cost.set_xlabel("Threshold")
        ax_cost.set_ylabel("Total Cost")
        ax_cost.set_title("Cost-Sensitive Threshold Optimization")
        ax_cost.grid(True)
        
        plt.tight_layout()
        plt.savefig('cost_sensitive_threshold_plot.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(cost_fig)

        # Distribution of Predicted Probabilities
        fig_dist, ax_dist = plt.subplots(figsize=(10, 6))
        sns.histplot(y_pred_proba_fraud[y_test_fraud == 0], color='blue', label='Non-Fraud (Actual 0)', 
                     kde=True, stat='density', alpha=0.5, ax=ax_dist)
        sns.histplot(y_pred_proba_fraud[y_test_fraud == 1], color='red', label='Fraud (Actual 1)', 
                     kde=True, stat='density', alpha=0.5, ax=ax_dist)
        ax_dist.set_title('Distribution of Predicted Fraud Probabilities (Test Set)')
        ax_dist.set_xlabel('Predicted Probability of Fraud')
        ax_dist.set_ylabel('Density')
        ax_dist.axvline(optimal_threshold, color='green', linestyle='--', label=f'Optimal Threshold ({optimal_threshold:.3f})')
        ax_dist.legend()
        ax_dist.grid(True)
        plt.tight_layout()
        plt.savefig('predicted_probabilities_distribution_final.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig_dist)

        # Enhanced Evaluation Metrics Bar Chart
        metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-score','PR-AUC']
        metrics_values = [accuracy_score_total, precision_score_fraud, recall_score_fraud, f1_score_fraud,pr_auc]

        plt.figure(figsize=(10, 7), dpi=120)
        ax = plt.gca()

        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
        bars = ax.bar(metrics_names, metrics_values, color=colors,
                      edgecolor='black', linewidth=1.2, alpha=0.85, width=0.6)

        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height - 0.05,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold',
                    color='white', bbox=dict(facecolor='black', alpha=0.7, pad=2))

        ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.7)
        ax.text(3.5, 0.82, 'Good Performance Threshold', fontsize=10, color='gray')

        ax.set_ylim(0, 1.15)
        ax.set_title("Final Fraud Detection Model Evaluation Metrics (Test Set)",
                     fontsize=16, pad=20, fontweight='bold')
        ax.set_ylabel('Score', fontsize=13, labelpad=10)
        ax.set_xlabel('Metric', fontsize=13, labelpad=10)

        ax.tick_params(axis='x', labelsize=12, pad=8)
        ax.tick_params(axis='y', labelsize=11)
        ax.grid(axis='y', linestyle=':', alpha=0.6)

        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        for spine in ['left', 'bottom']:
            ax.spines[spine].set_linewidth(1.5)

        plt.figtext(0.5, -0.1,
                    f"Model: CNN | Fraud Detection Rate: {recall_score_fraud:.1%} | Precision: {precision_score_fraud:.1%}",
                    ha="center", fontsize=11, bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

        plt.figtext(0.95, 0.05, "Fraud Analytics",
                    fontsize=12, color='gray', alpha=0.5,
                    ha='right', va='bottom', rotation='vertical')

        plt.tight_layout()
        plt.savefig('evaluation_metrics_final.png', dpi=300, bbox_inches='tight', transparent=False)
        plt.show()
        
    except KeyboardInterrupt:
        print("\nSimulation stopped by user")
    except Exception as e:
        print(f"\nError during simulation: {str(e)}")

if __name__ == "__main__":
    main()
    